In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import pickle
import os
from matplotlib.ticker import FormatStrFormatter


plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 14



In [ ]:
np.random.seed(99)

In [ ]:
%run ./time_Lorenz_data_6time.ipynb
%run ./time_Lorenz_data_2time_2_3.ipynb
%run ./time_Lorenz_data_2time_3_5.ipynb
%run ./ESN.ipynb
%run ./Reconstruct.ipynb
N_washout = 100
N_train   = 10000
N_tstart  = 10000
N_test    = 2000
#c = 0.3
window = 3
window_length = 100

#gamma_list = np.array([[0.12, 0.23], [0.12, 0.23], [0.3, 0.33]])

---

In [ ]:
N_xs = [4]

window = 6
window_length = 100




Array_to_plot_1 = [[],[],[],[],[],[],[],[],[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]

    density = 0.2 

    A_list = []

    for _ in range(6):
        A = (np.random.rand(N_x, N_x) < density).astype(float)
        np.fill_diagonal(A, 0)
        A_list.append(A)

    A1, A2, A3, A4, A5, A6 = A_list

    X = gen_data_norm_6time(N_x, A1, A2, A3, A4, A5, A6, N_tstart+N_test, window*window_length, window_length, sigma_dyn = 1e-6, sigma_obs = 1e-6)

    Array_to_plot_1[0].append(A1)
    Array_to_plot_1[1].append(A2)
    Array_to_plot_1[2].append(A3)
    Array_to_plot_1[3].append(A4)
    Array_to_plot_1[4].append(A5)
    Array_to_plot_1[5].append(A6)

    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window_1 = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 6

   
    


    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment



    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet_1 = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet_1[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet_1), axis=0)
    #TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
    for t in range (test_window):       
        TDI_window_1.append(np.max(TDI[t]))
        Array_to_plot_1 [6 + t].append(TDI[t])
    #TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(np.mean(TDI))
    #TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(gated_max_softmax_entropy_score(TDI))
    
    
    
    
    plt.plot(TDI_window_1)
    plt.xticks(ticks= range(len(TDI_window_1)), labels = [i  for i in range(len(TDI_window_1))])


    save_dict_1 = {
    'TDI_window_1': TDI_window_1,
    'Array_to_plot_1': Array_to_plot_1,
    'TDInet_1': TDInet_1
    }

    save_dir = os.path.join('result', 'Lorzen_results')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'Lorzen_experiment_1.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_1, f)
        
    print(f"data has been saved to: {file_path}")
     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Lorzen_results', 'Lorzen_experiment_1.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window_1 = loaded_data['TDI_window_1']
Array_to_plot_1 = loaded_data['Array_to_plot_1']
TDInet_1 = loaded_data['TDInet_1']

In [ ]:
titles = ['Ground Truth t - 0','Ground Truth t - 100','Ground Truth t - 200','Ground Truth t - 300','Ground Truth t - 400','Ground Truth t - 500','Time Delay 0','Time Delay 100','Time Delay 200','Time Delay 300','Time Delay 400', 'Time Delay 500']
N_xs = [4]
test_window = 6
for i in range(len(N_xs)):
    N_x = N_xs[i]
    variables = np.arange(1, N_x+1, 1)
    seq = np.arange(0, N_x, math.ceil(N_x / 10))


    highlight_ratio = 0.40
    vmax = np.max(Array_to_plot_1[7:12])
    if vmax == 0:
        vmax = 1.0
    threshold = vmax * highlight_ratio

    
        #plot_array_1x2([Array_to_plot[0][i][:,:,j], Array_to_plot[1][i][j]], titles, variables, seq)
    plot_array_2x6_Coupled_Lorzen_3([Array_to_plot_1[0][i], Array_to_plot_1[1][i],Array_to_plot_1[2][i],Array_to_plot_1[3][i],Array_to_plot_1[4][i],Array_to_plot_1[5][i],Array_to_plot_1[6][i], Array_to_plot_1[7][i], Array_to_plot_1[8][i], Array_to_plot_1[9][i], Array_to_plot_1[10][i], Array_to_plot_1[11][i]], titles, variables, seq, threshold)
        #plot_array_1x2([Array_to_plot[0][i][j], Array_to_plot[1][i]], titles, variables, seq)

In [ ]:
x = [i for i in range(len(TDI_window_1))]
violin_data = [TDInet_1[:, t].max(axis=(1, 2)) for t in range(test_window)]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
vp = ax.violinplot(
    violin_data,
    positions=np.arange(1, test_window + 1),
    showmeans=True, showextrema=True, showmedians=False, bw_method=0.5
)

for body in vp['bodies']:
    body.set_facecolor('#f6e09d')
    body.set_edgecolor('#e64532')
    body.set_alpha(0.9)

for partname in ('cbars', 'cmins', 'cmaxes', 'cmeans'):
    if partname in vp:
        vp[partname].set_edgecolor('#e64532')
        vp[partname].set_linewidth(1.5)


ax.set_xticks(np.arange(1, test_window + 1), labels=[(i)*100 for i in range(len(TDI_window_1))])

y_min, y_max = ax.get_ylim()

new_yticks = np.linspace(0, y_max, 4)
ax.set_yticks(new_yticks)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))


plt.tight_layout()
plt.show()

    # Save the plot
filename = f"Coupled_Lorzen_LCC_3"
pathname = os.path.join('result', 'figure', f"{filename}.png")
fig.savefig(pathname, bbox_inches = 'tight', dpi=600)

---

In [ ]:
# network size
N_xs = [4]

# Probability of generating an edge
percentages = [0.5, 0.4, 0.25]

Array_to_plot_2 = [[],[],[],[],[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]
    p = percentages[nx]
    A1 = np.ones([N_x, N_x])
    for i in range(N_x):
        A1[i,i] = 0

    A1[0,1] = 0
    A1[1,2] = 0
    A1[2,3] = 0
    A2 = np.zeros([N_x, N_x])
    A2[0,1] = 1
    A2[1,2] = 1
    A2[2,3] = 1
    X = gen_data_norm_2time_3_5(N_x, A1, A2, N_tstart+N_test, window*window_length, window_length, sigma_dyn = 1e-6, sigma_obs = 1e-6)
    #X, A = gen_time_X_p(N_x, p)
    Array_to_plot_2[0].append(A2)
    Array_to_plot_2[1].append(A1)
    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window_2 = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 6

   
    



    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment

    

    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet_2 = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet_2[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet_2), axis=0)
    #TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
    for t in range (test_window):       
        TDI_window_2.append(np.max(TDI[t]))
        Array_to_plot_2 [2 + t].append(TDI[t])
    #TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(np.mean(TDI))
    #TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(gated_max_softmax_entropy_score(TDI))
    
    
    
    
    plt.plot(TDI_window_2)
    plt.xticks(ticks= range(len(TDI_window_2)), labels = [i  for i in range(len(TDI_window_2))])

    save_dict_2 = {
    'TDI_window_2': TDI_window_2,
    'Array_to_plot_2': Array_to_plot_2,
    'TDInet_2': TDInet_2

    }

    save_dir = os.path.join('result', 'Lorzen_results')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'Lorzen_experiment_2.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)
        
    print(f"data has been saved to: {file_path}")

     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Lorzen_results', 'Lorzen_experiment_2.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window_2 = loaded_data['TDI_window_2']
Array_to_plot_2 = loaded_data['Array_to_plot_2']
TDInet_2 = loaded_data['TDInet_2']


In [ ]:
titles = ['Ground Truth t - 300','Ground Truth t - 500','Time Delay 0','Time Delay 100','Time Delay 200','Time Delay 300','Time Delay 400', 'Time Delay 500']
N_xs = [4]
test_window = 6
for i in range(len(N_xs)):
    N_x = N_xs[i]
    variables = np.arange(1, N_x+1, 1)
    seq = np.arange(0, N_x, math.ceil(N_x / 10))

    highlight_ratio = 0.40
    vmax = np.max(Array_to_plot_2[2:7])
    if vmax == 0:
        vmax = 1.0
    threshold = vmax * highlight_ratio

        #plot_array_1x2([Array_to_plot[0][i][:,:,j], Array_to_plot[1][i][j]], titles, variables, seq)
    plot_array_2x4_Coupled_Lorzen_2([Array_to_plot_2[0][i], Array_to_plot_2[1][i],Array_to_plot_2[2][i],Array_to_plot_2[3][i],Array_to_plot_2[4][i],Array_to_plot_2[5][i],Array_to_plot_2[6][i], Array_to_plot_2[7][i]], titles, variables, seq, threshold)
        #plot_array_1x2([Array_to_plot[0][i][j], Array_to_plot[1][i]], titles, variables, seq)

In [ ]:
x = [i for i in range(len(TDI_window_2))]
violin_data = [TDInet_2[:, t].max(axis=(1, 2)) for t in range(test_window)]

fig, ax = plt.subplots(1, 1, figsize=(8, 4))


vp = ax.violinplot(
    violin_data,
    positions=np.arange(1, test_window + 1),
    showmeans=True, showextrema=True, showmedians=False, bw_method=0.5
)


for body in vp['bodies']:
    body.set_facecolor('#fad7d3')
    body.set_edgecolor('#b53289')
    body.set_alpha(0.9)

for partname in ('cbars', 'cmins', 'cmaxes', 'cmeans'):
    if partname in vp:
        vp[partname].set_edgecolor('#b53289')
        vp[partname].set_linewidth(1.5)

threshold_lines = [0.0001, 0.0007]  

for y in threshold_lines:
    ax.axhline(
        y=y,
        color='red',
        linestyle='--',
        linewidth=1.8,
        alpha=0.85,
        zorder=0   
    )

ax.set_xticks(np.arange(1, test_window + 1), labels=[(i)*100 for i in range(len(TDI_window_2))])


y_min, y_max = ax.get_ylim()


new_yticks = np.linspace(0, y_max, 4)
ax.set_yticks(new_yticks)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))

plt.tight_layout()
plt.show()

    # Save the plot
filename = f"Coupled_Lorzen_LCC_2"
pathname = os.path.join('result', 'figure', f"{filename}.png")
fig.savefig(pathname, bbox_inches = 'tight', dpi=600)

---

In [ ]:
# network size
N_xs = [4]

# Probability of generating an edge
percentages = [0.5, 0.4, 0.25]

Array_to_plot = [[],[],[],[],[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]
    p = percentages[nx]
    A1 = np.ones([N_x, N_x])
    for i in range(N_x):
        A1[i,i] = 0

    A1[0,1] = 0
    A1[1,2] = 0
    A1[2,3] = 0
    
    A2 = np.ones([N_x, N_x])
    for i in range(N_x):
        A2[i,i] = 0

    A2[0,1] = 0
    A2[1,2] = 0
    A2[2,3] = 0
    X = gen_data_norm_2time_2_3(N_x, A1, A2, N_tstart+N_test, window*window_length, window_length, sigma_dyn = 1e-6, sigma_obs = 1e-6)
    #X, A = gen_time_X_p(N_x, p)
    Array_to_plot[0].append(A2)
    Array_to_plot[1].append(A1)
    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 6

   
    



    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment

    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet), axis=0)
    #TDI_window.append(np.max(TDI)*(softmax_entropy_score(TDI)+ 1e-6))
    for t in range (test_window ):       
        TDI_window.append(np.max(TDI[t]))
        Array_to_plot [2 + t].append(TDI[t])
    #TDI_window.append(np.std(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(np.mean(TDI))
    #TDI_window.append(np.max(TDI)/(np.mean(TDI) + 1e-6))
    #TDI_window.append(gated_max_softmax_entropy_score(TDI))
    
    
    
    
    plt.plot(TDI_window)
    plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])


    save_dict_2 = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet
    }

    save_dir = os.path.join('result', 'Lorzen_results')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'Lorzen_experiment.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)
        
    print(f"data has been saved to: {file_path}")

     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Lorzen_results', 'Lorzen_experiment.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']



In [ ]:
titles = ['Ground Truth t - 200','Ground Truth t - 300','Time Delay 0','Time Delay 100','Time Delay 200','Time Delay 300','Time Delay 400', 'Time Delay 500']
N_xs = [4]

test_window = 6
for i in range(len(N_xs)):
    N_x = N_xs[i]
    variables = np.arange(1, N_x+1, 1)
    seq = np.arange(0, N_x, math.ceil(N_x / 10))

    highlight_ratio = 0.40
    vmax = np.max(Array_to_plot[2:7])
    if vmax == 0:
        vmax = 1.0
    threshold = vmax * highlight_ratio

    
        #plot_array_1x2([Array_to_plot[0][i][:,:,j], Array_to_plot[1][i][j]], titles, variables, seq)
    plot_array_2x4_Coupled_Lorzen_1([Array_to_plot[0][i], Array_to_plot[1][i],Array_to_plot[2][i],Array_to_plot[3][i],Array_to_plot[4][i],Array_to_plot[5][i],Array_to_plot[6][i], Array_to_plot[7][i]], titles, variables, seq, threshold)
        #plot_array_1x2([Array_to_plot[0][i][j], Array_to_plot[1][i]], titles, variables, seq)

In [ ]:
x = [i for i in range(len(TDI_window))]
violin_data = [TDInet[:, t].max(axis=(1, 2)) for t in range(test_window)]


fig, ax = plt.subplots(1, 1, figsize=(8, 4))
vp = ax.violinplot(
    violin_data,
    positions=np.arange(1, test_window + 1),
    showmeans=True, showextrema=True, showmedians=False, bw_method=0.5
)

for body in vp['bodies']:
    body.set_facecolor('#cfeadf')
    body.set_edgecolor('#4d7e54') 
    body.set_alpha(0.9)           


for partname in ('cbars', 'cmins', 'cmaxes', 'cmeans'):
    if partname in vp:
        vp[partname].set_edgecolor('#4d7e54')
        vp[partname].set_linewidth(1.5)


ax.set_xticks(np.arange(1, test_window + 1), labels=[(i)*100 for i in range(len(TDI_window))])


threshold_lines = [0.0005, 0.00108]  

for y in threshold_lines:
    ax.axhline(
        y=y,
        color='red',
        linestyle='--',
        linewidth=1.8,
        alpha=0.85,
        zorder=0   
    )


y_min, y_max = ax.get_ylim()


new_yticks = np.linspace(0, y_max, 4)
ax.set_yticks(new_yticks)
ax.yaxis.set_major_formatter(FormatStrFormatter('%.4f'))

plt.tight_layout()
plt.show()

    # Save the plot
filename = f"Coupled_Lorzen_LCC_1"
pathname = os.path.join('result', 'figure', f"{filename}.png")
fig.savefig(pathname, bbox_inches = 'tight', dpi=600)

---

In [ ]:
#Further Experiment



In [ ]:
# network size
N_xs = [4]


window_length = 100

window = 3


# Probability of generating an edge
percentages = [0.5, 0.4, 0.25]

Array_to_plot = [[],[],[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]
    p = percentages[nx]
    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1
    A2 = np.zeros([N_x, N_x])
    X = gen_data_norm_2time_2_3(N_x, A1, A2, N_tstart+N_test, window*100, 100, sigma_dyn = 1e-6, sigma_obs = 1e-6)

    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 6

   
    

#     X_segments = np.zeros((T - test_window*window_length + 1, test_window * D))

#     for i in range(T - test_window*window_length + 1):
#         segment = X[i:i + test_window*window_length:window_length].reshape(-1)  
#         X_segments[i] = segment


    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment



    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet), axis=0)

    for t in range (test_window ):       
        TDI_window.append(np.max(TDI[t]))
        Array_to_plot [t].append(TDI[t])

    
    
    
    plt.plot(TDI_window)
    plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])


    save_dict_2 = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet
    }

    save_dir = os.path.join('result', 'Further_experiment')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'L=100.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)
        
    print(f"data has been saved to: {file_path}")

     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Further_experiment', 'L=100.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']


In [ ]:
variables = np.arange(1, 5, 1)
seq = np.arange(0, len(variables), 1)

highlight_ratio = 0.40


vmax = np.max(Array_to_plot)
threshold = max(vmax * highlight_ratio, 0.001)

titles = [f"Time Delay {100*(i)}" for i in range(6)]


plot_array_1x4_Further_experiment([Array_to_plot[0][0], Array_to_plot[1][0],Array_to_plot[2][0],Array_to_plot[3][0],Array_to_plot[4][0],Array_to_plot[5][0]], Array_to_plot, titles, variables, seq, threshold)

In [ ]:
# network size
N_xs = [4]

N_washout = 100
N_train   = 9000
N_tstart  = 9000
N_test    = 2000
N_pro     = 13000

window = 3


window_length = 200

# Probability of generating an edge

Array_to_plot = [[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]

    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1
    A2 = np.zeros([N_x, N_x])
    X = gen_data_norm_2time_2_3(N_x, A1, A2, N_pro+N_test, window*100, 100, sigma_dyn = 1e-6, sigma_obs = 1e-6)

    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 4

   
    

#     X_segments = np.zeros((T - test_window*window_length + 1, test_window * D))

#     for i in range(T - test_window*window_length + 1):
#         segment = X[i:i + test_window*window_length:window_length].reshape(-1)  
#         X_segments[i] = segment



    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment



    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet), axis=0)

    for t in range (test_window ):       
        TDI_window.append(np.max(TDI[t]))
        Array_to_plot [t].append(TDI[t])

    
    
    
    plt.plot(TDI_window)
    plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])


    save_dict_2 = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet
    }

    save_dir = os.path.join('result', 'Further_experiment')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'L=200.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)
        
    print(f"data has been saved to: {file_path}")

     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Further_experiment', 'L=200.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']


In [ ]:
variables = np.arange(1, 5, 1)
seq = np.arange(0, len(variables), 1)

highlight_ratio = 0.40

Arrays_selected = Array_to_plot

vmax = np.max(Array_to_plot)
color_bar = 0.0013
threshold = max(vmax * highlight_ratio, color_bar * highlight_ratio)

titles = [f"Time Delay {200*(i)}" for i in range(4)]
# Time Delay 200, 400, 600, 800

plot_array_1x4_Further_experiment_2(
    [Arrays_selected[0][0],
     Arrays_selected[1][0],
     Arrays_selected[2][0],
     Arrays_selected[3][0]], Array_to_plot,
    titles,
    variables,
    seq,
    threshold,
    color_bar
)

In [ ]:
# network size
N_xs = [4]
window = 3
N_washout = 100
N_train   = 9000
N_tstart  = 9000
N_test    = 2000
N_pro     = 13000
window_length = 290

# Probability of generating an edge
percentages = [0.5, 0.4, 0.25]

Array_to_plot = [[],[],[],[]]
labels=['IRC']
for nx in range(len(N_xs)):
    N_x = N_xs[nx]
    p = percentages[nx]
    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1
    A2 = np.zeros([N_x, N_x])
    X = gen_data_norm_2time_2_3(N_x, A1, A2, N_pro + N_test, window*100, 100, sigma_dyn = 1e-6, sigma_obs = 1e-6)

    

    # washout
    X_washout = X[:N_washout]
    # training
    X_t = X[N_washout:N_washout+N_train-1]
    Y_t = X[N_washout+1:N_washout+N_train]

    X_test = X[N_tstart:]
    N_dim   = 3 * N_x # dimension of inputs (and outputs)
    N_units = 100 * N_dim #units in the reservoir 

    N_net = 20
 
    
    T, D = X.shape  

    k = 0   
    TDI_window = []
    N_rep, N_evo = 30, 5

    offset = np.zeros([N_net, N_rep])
    for s in range(N_net):
        for rep in range(N_rep):
            offset[s,rep] = np.random.randint(X_test.shape[0] - N_evo - N_washout - window -1)

    test_window = 4

   
    

    X_segments = np.zeros((T - (test_window-1)*window_length , test_window * D))

    for i in range((test_window - 1) * window_length, T):
        segment = X[i - np.arange(test_window) * window_length].reshape(-1)
        X_segments[i - (test_window - 1) * window_length] = segment

    N_washout_segments = N_washout
    N_train_segments   = N_train
    N_tstart_segments  = N_tstart
    N_test_segments    = T-N_train_segments


    N_dim_segments = test_window*N_dim # dimension of inputs (and outputs)
    N_units_segments = 100 * N_dim_segments #units in the reservoir 

# washout
    X_washout_segments= X_segments[:N_washout_segments]
# training
    X_t_segments = X_segments[N_washout_segments:N_washout_segments+N_train_segments-1]
    Y_t_segments = X_segments[N_washout_segments+1:N_washout_segments+N_train_segments]

    X_test_segments = X_segments[N_tstart_segments:]

    TDInet = np.zeros([N_net,test_window,N_x,N_x])






    for net in range(N_net):
        esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
        esn.train(X_washout_segments, X_t_segments, Y_t_segments)

    #esn = ESN(N_units, N_dim, N_dim)
    #esn.train(X_washout, X_t, Y_t)

    
        for t_idx in range (0, test_window):
    #TDInet.append(cal_time_TDI(esn, X_test_segments, N_x, 3, N_dim, window, 0, 1,  N_washout, N_evo))
            TDInet[net, t_idx] = cal_time_TDI_copy_detect(esn, X_test_segments, N_x, 3, N_dim, N_rep, test_window, 0, 1,  N_washout, N_evo, offset[net], t_idx)
    #TDInet.append(cal_TDI(esn, X_test, N_x, 3, 0, 1, 20, N_washout, N_evo))


    TDI = np.mean(np.array(TDInet), axis=0)

    for t in range (test_window ):       
        TDI_window.append(np.max(TDI[t]))
        Array_to_plot [t].append(TDI[t])

    
    
    
    plt.plot(TDI_window)
    plt.xticks(ticks= range(len(TDI_window)), labels = [i for i in range(len(TDI_window))])


    save_dict_2 = {
    'TDI_window': TDI_window,
    'Array_to_plot': Array_to_plot,
    'TDInet': TDInet
    }

    save_dir = os.path.join('result', 'Further_experiment')

    os.makedirs(save_dir, exist_ok=True) 

    file_path = os.path.join(save_dir, 'L=290.pkl')


    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)
        
    print(f"data has been saved to: {file_path}")

     



In [ ]:
%run ./Plot_DDI.ipynb

file_path = os.path.join('result', 'Further_experiment', 'L=290.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

TDI_window = loaded_data['TDI_window']
Array_to_plot = loaded_data['Array_to_plot']
TDInet = loaded_data['TDInet']


In [ ]:
variables = np.arange(1, 5, 1)
seq = np.arange(0, len(variables), 1)

highlight_ratio = 0.40
Arrays_selected = Array_to_plot

vmax = np.max(Array_to_plot)
color_bar = 0.0013
threshold = max(vmax * highlight_ratio, color_bar * highlight_ratio)


titles = [f"Time Delay {290*(i)}" for i in range(4)]


plot_array_1x4_Further_experiment_3(
    [Arrays_selected[0][0],
     Arrays_selected[1][0],
     Arrays_selected[2][0],
     Arrays_selected[3][0]], Array_to_plot,
    titles,
    variables,
    seq,
    threshold,
    color_bar
)

---

In [ ]:
#Further Experiment_2


In [ ]:
N_xs = [4]

window = 3

N_washout = 100
N_train   = 9000
N_tstart  = 9000
N_test    = 2000
N_pro     = 13000

true_lag = 300

lag_min = 285
lag_max = 315
lag_step = 5

lag_list = [0] + list(np.arange(lag_min, lag_max + lag_step, lag_step))
lag_labels = ["t"] + [f"t-{lag}" for lag in lag_list[1:]]

test_window = len(lag_list)

percentages = [0.5, 0.4, 0.25]
labels = ['IRC']

for nx in range(len(N_xs)):

    N_x = N_xs[nx]
    p = percentages[nx]

    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1

    A2 = np.zeros([N_x, N_x])

    X = gen_data_norm_2time_2_3(
        N_x,
        A1,
        A2,
        N_pro + N_test,
        true_lag,
        100,
        sigma_dyn=1e-6,
        sigma_obs=1e-6
    )

    T_total, D = X.shape

    # Lorenz: each node has 3 variables
    n_nodes_dim = 3
    N_dim = D
    N_units = 100 * N_dim

    N_net = 20
    N_rep, N_evo = 30, 5

    # Construct high-dimensional spatiotemporal sequence
    # X_segments[t] = [X(t), X(t-280), X(t-290), X(t-300), X(t-310)]
    max_lag = max(lag_list)

    X_segments = np.zeros(
        (T_total - max_lag, test_window * D)
    )

    for row, t in enumerate(range(max_lag, T_total)):
        segment = np.concatenate(
            [X[t - lag] for lag in lag_list],
            axis=0
        )
        X_segments[row] = segment

    # Segment train/test split
    N_washout_segments = N_washout
    N_train_segments = N_train
    N_tstart_segments = N_tstart

    X_washout_segments = X_segments[:N_washout_segments]

    X_t_segments = X_segments[
        N_washout_segments : N_washout_segments + N_train_segments - 1
    ]

    Y_t_segments = X_segments[
        N_washout_segments + 1 : N_washout_segments + N_train_segments
    ]

    X_test_segments = X_segments[N_tstart_segments:]

    # ESN dimensions
    N_dim_segments = test_window * N_dim
    N_units_segments = 100 * N_dim_segments

    # Offset
    max_offset = X_test_segments.shape[0] - N_evo - 1

    if max_offset <= N_washout:
        raise ValueError(
            f"Insufficient offset range: max_offset={max_offset}, "
            f"N_washout={N_washout}. Increase N_test or reduce N_washout/N_evo."
        )

    offset = np.random.randint(
        N_washout,
        max_offset,
        size=(N_net, N_rep)
    )

    TDInet = np.zeros([N_net, test_window, N_x, N_x])

    for net in range(N_net):

        esn = ESN(
            N_units_segments,
            N_dim_segments,
            N_dim_segments
        )

        esn.train(
            X_washout_segments,
            X_t_segments,
            Y_t_segments
        )

        for t_idx in range(test_window):

            TDInet[net, t_idx] = cal_time_TDI_copy_detect(
                esn,
                X_test_segments,
                N_x,
                n_nodes_dim,
                N_dim,
                N_rep,
                test_window,
                0,              # target_i: effect variable x
                1,              # target_j: cause variable y
                N_washout,
                N_evo,
                offset[net],
                t_idx
            )

    TDI = np.mean(TDInet, axis=0)

    TDI_window = []
    Array_to_plot = []

    for t_idx, lag in enumerate(lag_list):

        TDI_block = TDI[t_idx]

        TDI_window.append(np.max(TDI_block))

        Array_to_plot.append(TDI_block.copy())


    save_dict_2 = {
        'lag_list': lag_list,
        'lag_labels': lag_labels,
        'true_lag': true_lag,
        'test_window': test_window,
        'TDI_window': TDI_window,
        'Array_to_plot': Array_to_plot,
        'TDI': TDI,
        'TDInet': TDInet,

        'config': {
            'N_x': N_x,
            'N_washout': N_washout,
            'N_train': N_train,
            'N_tstart': N_tstart,
            'N_test': N_test,
            'N_pro': N_pro,
            'N_net': N_net,
            'N_rep': N_rep,
            'N_evo': N_evo,
            'true_lag': true_lag,
            'lag_list': lag_list,
            'n_nodes_dim': n_nodes_dim,
            'N_dim': N_dim,
            'N_dim_segments': N_dim_segments,
            'N_units_segments': N_units_segments
        }
    }

    save_dir = os.path.join('result', 'Further_experiment')
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(
        save_dir,
        'multi_lag_blocks_0_280_290_300_310.pkl'
    )

    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)

    print(f"data has been saved to: {file_path}")

In [ ]:
%run ./Plot_DDI.ipynb
file_path = os.path.join('result', 'Further_experiment', 'multi_lag_blocks_0_280_290_300_310.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

lag_list = loaded_data['lag_list']
TDI_window = loaded_data['TDI_window']

plot_bar_windowlength_Further_experiment(
    lag_list[1:],
    TDI_window[1:],
    "Further_experiment_4_dt_001",'#a9bfd3','#1f3b5b'
)

In [ ]:
N_xs = [4]

window = 3

N_washout = 100
N_train   = 9000
N_tstart  = 9000
N_test    = 2000
N_pro     = 13000

true_lag = 300

lag_min = 285
lag_max = 315
lag_step = 5

lag_list = [0] + list(np.arange(lag_min, lag_max + lag_step, lag_step))
lag_labels = ["t"] + [f"t-{lag}" for lag in lag_list[1:]]

test_window = len(lag_list)

percentages = [0.5, 0.4, 0.25]
labels = ['IRC']

for nx in range(len(N_xs)):

    N_x = N_xs[nx]
    p = percentages[nx]

    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1

    A2 = np.zeros([N_x, N_x])

    X = gen_data_norm_2time_2_3(
        N_x,
        A1,
        A2,
        N_pro + N_test,
        true_lag,
        100,
        sigma_dyn=1e-6,
        sigma_obs=1e-6, dt = 0.02
    )

    T_total, D = X.shape

    # Lorenz: each node has 3 variables
    n_nodes_dim = 3
    N_dim = D
    N_units = 100 * N_dim

    N_net = 20
    N_rep, N_evo = 30, 5

    # Construct high-dimensional spatiotemporal sequence
    # X_segments[t] = [X(t), X(t-280), X(t-290), X(t-300), X(t-310)]
    max_lag = max(lag_list)

    X_segments = np.zeros(
        (T_total - max_lag, test_window * D)
    )

    for row, t in enumerate(range(max_lag, T_total)):
        segment = np.concatenate(
            [X[t - lag] for lag in lag_list],
            axis=0
        )
        X_segments[row] = segment

    # Segment train/test split
    N_washout_segments = N_washout
    N_train_segments = N_train
    N_tstart_segments = N_tstart

    X_washout_segments = X_segments[:N_washout_segments]

    X_t_segments = X_segments[
        N_washout_segments : N_washout_segments + N_train_segments - 1
    ]

    Y_t_segments = X_segments[
        N_washout_segments + 1 : N_washout_segments + N_train_segments
    ]

    X_test_segments = X_segments[N_tstart_segments:]

    # ESN dimensions
    N_dim_segments = test_window * N_dim
    N_units_segments = 100 * N_dim_segments

    # Offset
    max_offset = X_test_segments.shape[0] - N_evo - 1

    if max_offset <= N_washout:
        raise ValueError(
            f"Insufficient offset range: max_offset={max_offset}, "
            f"N_washout={N_washout}. Increase N_test or reduce N_washout/N_evo."
        )

    offset = np.random.randint(
        N_washout,
        max_offset,
        size=(N_net, N_rep)
    )

    TDInet = np.zeros([N_net, test_window, N_x, N_x])

    for net in range(N_net):

        esn = ESN(
            N_units_segments,
            N_dim_segments,
            N_dim_segments
        )

        esn.train(
            X_washout_segments,
            X_t_segments,
            Y_t_segments
        )

        for t_idx in range(test_window):

            TDInet[net, t_idx] = cal_time_TDI_copy_detect(
                esn,
                X_test_segments,
                N_x,
                n_nodes_dim,
                N_dim,
                N_rep,
                test_window,
                0,              # target_i: effect variable x
                1,              # target_j: cause variable y
                N_washout,
                N_evo,
                offset[net],
                t_idx
            )

    TDI = np.mean(TDInet, axis=0)

    TDI_window = []
    Array_to_plot = []

    for t_idx, lag in enumerate(lag_list):

        TDI_block = TDI[t_idx]

        TDI_window.append(np.max(TDI_block))

        Array_to_plot.append(TDI_block.copy())


    save_dict_2 = {
        'lag_list': lag_list,
        'lag_labels': lag_labels,
        'true_lag': true_lag,
        'test_window': test_window,
        'TDI_window': TDI_window,
        'Array_to_plot': Array_to_plot,
        'TDI': TDI,
        'TDInet': TDInet,

        'config': {
            'N_x': N_x,
            'N_washout': N_washout,
            'N_train': N_train,
            'N_tstart': N_tstart,
            'N_test': N_test,
            'N_pro': N_pro,
            'N_net': N_net,
            'N_rep': N_rep,
            'N_evo': N_evo,
            'true_lag': true_lag,
            'lag_list': lag_list,
            'n_nodes_dim': n_nodes_dim,
            'N_dim': N_dim,
            'N_dim_segments': N_dim_segments,
            'N_units_segments': N_units_segments
        }
    }

    save_dir = os.path.join('result', 'Further_experiment')
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(
        save_dir,
        'multi_lag_blocks_0_280_290_300_310_dt_002.pkl'
    )

    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)

    print(f"data has been saved to: {file_path}")

In [ ]:
%run ./Plot_DDI.ipynb
file_path = os.path.join('result', 'Further_experiment', 'multi_lag_blocks_0_280_290_300_310_dt_002.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

lag_list = loaded_data['lag_list']
TDI_window = loaded_data['TDI_window']

plot_bar_windowlength_Further_experiment(
    lag_list[1:],
    TDI_window[1:],
    "Further_experiment_4_dt_002","#FEDCAE","#FF7F00"
)

In [ ]:
N_xs = [4]

window = 3

N_washout = 100
N_train   = 9000
N_tstart  = 9000
N_test    = 2000
N_pro     = 13000

true_lag = 300

lag_min = 285
lag_max = 315
lag_step = 5

lag_list = [0] + list(np.arange(lag_min, lag_max + lag_step, lag_step))
lag_labels = ["t"] + [f"t-{lag}" for lag in lag_list[1:]]

test_window = len(lag_list)

percentages = [0.5, 0.4, 0.25]
labels = ['IRC']

for nx in range(len(N_xs)):

    N_x = N_xs[nx]
    p = percentages[nx]

    A1 = np.zeros([N_x, N_x])
    A1[0, 1] = 1

    A2 = np.zeros([N_x, N_x])

    X = gen_data_norm_2time_2_3(
        N_x,
        A1,
        A2,
        N_pro + N_test,
        true_lag,
        100,
        sigma_dyn=1e-6,
        sigma_obs=1e-6, dt = 0.025
    )

    T_total, D = X.shape

    # Lorenz: each node has 3 variables
    n_nodes_dim = 3
    N_dim = D
    N_units = 100 * N_dim

    N_net = 20
    N_rep, N_evo = 30, 5

    # Construct high-dimensional spatiotemporal sequence
    # X_segments[t] = [X(t), X(t-280), X(t-290), X(t-300), X(t-310)]
    max_lag = max(lag_list)

    X_segments = np.zeros(
        (T_total - max_lag, test_window * D)
    )

    for row, t in enumerate(range(max_lag, T_total)):
        segment = np.concatenate(
            [X[t - lag] for lag in lag_list],
            axis=0
        )
        X_segments[row] = segment

    # Segment train/test split
    N_washout_segments = N_washout
    N_train_segments = N_train
    N_tstart_segments = N_tstart

    X_washout_segments = X_segments[:N_washout_segments]

    X_t_segments = X_segments[
        N_washout_segments : N_washout_segments + N_train_segments - 1
    ]

    Y_t_segments = X_segments[
        N_washout_segments + 1 : N_washout_segments + N_train_segments
    ]

    X_test_segments = X_segments[N_tstart_segments:]

    # ESN dimensions
    N_dim_segments = test_window * N_dim
    N_units_segments = 100 * N_dim_segments

    # Offset
    max_offset = X_test_segments.shape[0] - N_evo - 1

    if max_offset <= N_washout:
        raise ValueError(
            f"Insufficient offset range: max_offset={max_offset}, "
            f"N_washout={N_washout}. Increase N_test or reduce N_washout/N_evo."
        )

    offset = np.random.randint(
        N_washout,
        max_offset,
        size=(N_net, N_rep)
    )

    TDInet = np.zeros([N_net, test_window, N_x, N_x])

    for net in range(N_net):

        esn = ESN(
            N_units_segments,
            N_dim_segments,
            N_dim_segments
        )

        esn.train(
            X_washout_segments,
            X_t_segments,
            Y_t_segments
        )

        for t_idx in range(test_window):

            TDInet[net, t_idx] = cal_time_TDI_copy_detect(
                esn,
                X_test_segments,
                N_x,
                n_nodes_dim,
                N_dim,
                N_rep,
                test_window,
                0,              # target_i: effect variable x
                1,              # target_j: cause variable y
                N_washout,
                N_evo,
                offset[net],
                t_idx
            )

    TDI = np.mean(TDInet, axis=0)

    TDI_window = []
    Array_to_plot = []

    for t_idx, lag in enumerate(lag_list):

        TDI_block = TDI[t_idx]

        TDI_window.append(np.max(TDI_block))

        Array_to_plot.append(TDI_block.copy())


    save_dict_2 = {
        'lag_list': lag_list,
        'lag_labels': lag_labels,
        'true_lag': true_lag,
        'test_window': test_window,
        'TDI_window': TDI_window,
        'Array_to_plot': Array_to_plot,
        'TDI': TDI,
        'TDInet': TDInet,

        'config': {
            'N_x': N_x,
            'N_washout': N_washout,
            'N_train': N_train,
            'N_tstart': N_tstart,
            'N_test': N_test,
            'N_pro': N_pro,
            'N_net': N_net,
            'N_rep': N_rep,
            'N_evo': N_evo,
            'true_lag': true_lag,
            'lag_list': lag_list,
            'n_nodes_dim': n_nodes_dim,
            'N_dim': N_dim,
            'N_dim_segments': N_dim_segments,
            'N_units_segments': N_units_segments
        }
    }

    save_dir = os.path.join('result', 'Further_experiment')
    os.makedirs(save_dir, exist_ok=True)

    file_path = os.path.join(
        save_dir,
        'multi_lag_blocks_0_280_290_300_310_dt_0025.pkl'
    )

    with open(file_path, 'wb') as f:
        pickle.dump(save_dict_2, f)

    print(f"data has been saved to: {file_path}")

In [ ]:
%run ./Plot_DDI.ipynb
file_path = os.path.join('result', 'Further_experiment', 'multi_lag_blocks_0_280_290_300_310_dt_0025.pkl')

with open(file_path, 'rb') as f:
    loaded_data = pickle.load(f)

lag_list = loaded_data['lag_list']
TDI_window = loaded_data['TDI_window']

plot_bar_windowlength_Further_experiment(
    lag_list[1:],
    TDI_window[1:],
    "Further_experiment_4_dt_0025",'#fad7d3','#b53289'
)